# Pharmacy Dataset Cleaning

Clean, standardize, validate, and save the messy pharmacy prescription dataset.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

input_path = "Pharmacy_Messy_31.csv"

df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (31, 7)


## Standardize Text and Missing Values

In [15]:
str_cols = [
    col for col in df.columns
    if pd.api.types.is_string_dtype(df[col])
]
for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in str_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
PrescriptionID    0
PatientID         1
DrugName          0
Dose              0
Quantity          0
Frequency         0
Status            0
dtype: int64


## Normalize Prescription Fields

In [ ]:
df["DrugName"] = df["DrugName"].str.title()

print("Drug names:", sorted(df["DrugName"].dropna().unique()))

Statuses: ['Active', 'Completed']
Frequencies: ['As needed', 'Every 8 hours', 'Once daily', 'Thrice daily', 'Twice daily']
Patient IDs with missing values: 1


In [ ]:
df["Status"] = df["Status"].str.title()

print("Statuses:", sorted(df["Status"].dropna().unique()))

In [ ]:
frequency_map = {
    "Once Daily": "Once daily",
    "Twice Daily": "Twice daily",
    "Thrice Daily": "Thrice daily",
    "Every 8 Hours": "Every 8 hours",
    "As Needed": "As needed",
}
df["Frequency"] = df["Frequency"].str.title().map(frequency_map)

print("Frequencies:", sorted(df["Frequency"].dropna().unique()))

In [ ]:
df["PatientID"] = df["PatientID"].str.upper()
df["PatientID"] = df["PatientID"].replace({"302": "PAT302", "PATT301": "PAT301"})

print("Patient IDs with missing values:", int(df["PatientID"].isna().sum()))

## Clean Dose and Quantity

In [ ]:
df["Dose_mg"] = (
    df["Dose"]
    .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False)
    .astype("float64")
)
invalid_dose_count = df["Dose_mg"].isna().sum()
dose_medians = df.groupby("DrugName")["Dose_mg"].transform("median")
df["Dose_mg"] = df["Dose_mg"].fillna(dose_medians)

print("Invalid doses imputed:", invalid_dose_count)
print(df.groupby("DrugName")["Dose_mg"].agg(["min", "max"]))

Invalid doses imputed: 1
Dose values by drug:
                 min     max
DrugName                    
Amlodipine       5.0    10.0
Amoxicillin    250.0   500.0
Azithromycin   500.0   500.0
Cetirizine      10.0    10.0
Ibuprofen      400.0   400.0
Metformin     1000.0  1000.0
Omeprazole      20.0    20.0
Paracetamol    500.0   650.0
Quantity values after cleaning:
count    31.000000
mean     18.306452
std       6.933827
min      10.000000
25%      13.750000
50%      20.000000
75%      20.000000
max      30.000000
Name: Quantity, dtype: float64


In [ ]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan
quantity_medians = df.groupby("DrugName")["Quantity"].transform("median")
df["Quantity"] = df["Quantity"].fillna(quantity_medians)

print("Quantity values after cleaning:")
print(df["Quantity"].describe())

## Validate and Save Cleaned Dataset

In [18]:
allowed_statuses = {"Active", "Completed"}
allowed_frequencies = set(frequency_map.values())

assert df["PrescriptionID"].is_unique
assert df["PatientID"].dropna().str.fullmatch(r"PAT\d+").all()
assert df["Status"].isin(allowed_statuses).all()
assert df["Frequency"].isin(allowed_frequencies).all()
assert df["Dose_mg"].notna().all() and (df["Dose_mg"] > 0).all()
assert df["Quantity"].notna().all() and (df["Quantity"] > 0).all()

print("Validation passed")
print("Duplicate prescription IDs:", df["PrescriptionID"].duplicated().sum())
print("Missing values:")
print(df.isna().sum())

Validation passed
Duplicate prescription IDs: 0
Missing values:
PrescriptionID    0
PatientID         1
DrugName          0
Dose              0
Quantity          0
Frequency         0
Status            0
Dose_mg           0
dtype: int64


In [ ]:
df = df.drop(columns="Dose")
df = df.rename(columns={"Dose_mg": "Dose_mg"})

output_path = "Pharmacy_Cleaned_31.csv"
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)

print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\Pharmacy\Pharmacy_Cleaned_31.csv
Saved shape: (31, 7)
